In [30]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Annotated
import operator
from langchain_groq import ChatGroq
import os


In [31]:
class Worker(BaseModel):
    name:str
    dependency:list[str] = Field(default_factory=list)
    status:str='PENDING'



In [32]:
class Nodestate(BaseModel):
    completed_tasks: Annotated[
                list[str],
                operator.add
            ] = Field(
                default_factory=list
           )
    result:list[str]=[]
    workers:Annotated[list[Worker],operator.add]=Field(default_factory=list)

In [ ]:
def research_worker(state:Nodestate):
    return{
        state.name:"Research",
        state.dependency:[],
        state.status:"READY"
    }
def finance_review(state:Nodestate):
    return{
        state.name:"FinanceReview",
        state.dependency:[],
        state.status:"READY"
    }
def final_review(state:Nodestate):
    return{
        state.name:"FinalReview",
        state.dependency:['"Research"',"FinanceReview"],
        state.status:'PENDING'
    }
def writer(state:Nodestate):
    prompt="you are an expert review person review my above node"
    result=llm.invoke(prompt)
    return{
        state.dependency:['final_review'],
        state.result:result
    }
def schedular(state:Nodestate):
    for task in state.workers:
        if task.name in task.completed_tasks:
            continue
        dependancy_satisfied=all(
            dep in task.completed_tasks
            for dep in task.dependency
        )
        if dependancy_satisfied:
            task.status='RUNING'
        else:
            task.status='BLOCKED'
    return {
        "status":state.status
    }
def runtime(state:Nodestate):
    for task in state.workers:
        if task.status!="RUNNING":
            continue
        print(
                    f"Executing: {task.name}"
                )
        task.status="SUCCESS"
    completed_now.append(task.name)
    return{
        "Worker":state.name,
        "completed_tasks":completed_now
        }
def remaining_task(state:Nodestate):
    for task in state.workers:
        if task.name not in state.completed_tasks:
            return "Scheduler"
    return "END"

In [ ]:
graph=StateGraph(Nodestate)
graph.add_node("Rsearch",research_worker)
graph.add_node("FinanceReview",finance_review)
graph.add_node("Schedular",schedular)
graph.add_node("Runtime",runtime)
graph.add_edge(START,"Research")
graph.add_edge(START,"FinanceReview")
graph.add_edge("Research","Schedular")
graph.add_edge("FinanceReview","Schedular")
graph.add_edge("Schedular","Runtime")
graph.add_conditional_edges(
    "Runtime",
    remaining_task,
    {
                "Schedular": "Schedular",
                "END": END
    }
)
app = graph.compile()
app

ValueError: Found edge starting at unknown node 'Research'